# Minimal Forecasting Pipeline
# ============================

A self-contained Jupyter notebook demonstrating a basic time series forecasting pipeline.
Uses the classic AirPassengers dataset and ARIMA modeling.

Requirements: pandas, numpy, matplotlib, statsmodels

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load the classic AirPassengers dataset
from statsmodels.datasets import get_rdataset
data = get_rdataset('AirPassengers', 'datasets')
df = data.data

# Convert to time series with datetime index
df['time'] = pd.to_datetime(df['time'])
df.set_index('time', inplace=True)
df = df[['value']]
df.columns = ['passengers']

print(df.head())

## 2. Exploratory Data Analysis

In [ ]:
# Plot the time series
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['passengers'], label='Passengers')
plt.title('AirPassengers Time Series')
plt.xlabel('Date')
plt.ylabel('Number of Passengers')
plt.grid(True)
plt.legend()
plt.show()

# Check for stationarity - plot ACF and PACF
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(df['passengers'], lags=40, ax=ax1)
plot_pacf(df['passengers'], lags=40, ax=ax2, method='ywm')
plt.tight_layout()
plt.show()

## 3. Data Preparation

We'll difference the data to make it stationary and split into train/test sets.

In [ ]:
# First difference to remove trend
df['passengers_diff'] = df['passengers'].diff()
df.dropna(inplace=True)

# Seasonal difference (12 months)
df['passengers_seasonal_diff'] = df['passengers_diff'].diff(12)
df.dropna(inplace=True)

# Split into train and test (last 12 months for testing)
train = df.iloc[:-12]['passengers']
test = df.iloc[-12:]['passengers']

print(f"Train size: {len(train)}, Test size: {len(test)}")

## 4. Model Training (ARIMA)

We'll use ARIMA(p,d,q) where:
- p: AR order (from PACF)
- d: Differencing order
- q: MA order (from ACF)

In [ ]:
# Fit ARIMA model - using manual params for simplicity
# Based on ACF/PACF, we'll try ARIMA(1,1,1) with seasonal component (0,1,1,12)
model = ARIMA(train, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12))
model_fit = model.fit()

# Summary of the model
print(model_fit.summary())

## 5. Forecasting

In [ ]:
# Forecast for the test period
forecast_steps = len(test)
forecast = model_fit.get_forecast(steps=forecast_steps)
forecast_ci = forecast.conf_int()

# Get predicted values
forecast_values = forecast.predicted_mean

# Create date range for forecast
last_train_date = train.index[-1]
forecast_dates = pd.date_range(start=last_train_date + pd.DateOffset(months=1), periods=forecast_steps, freq='MS')

# Plot results
plt.figure(figsize=(12, 6))
plt.plot(train.index, train, label='Training Data')
plt.plot(test.index, test, label='Actual Test Data', color='green')
plt.plot(forecast_dates, forecast_values, label='Forecast', color='red')
plt.fill_between(forecast_dates, 
                 forecast_ci.iloc[:, 0], 
                 forecast_ci.iloc[:, 1], 
                 color='pink', alpha=0.3)
plt.title('ARIMA Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Passengers')
plt.legend()
plt.grid(True)
plt.show()

## 6. Evaluation

In [ ]:
# Calculate metrics
mae = mean_absolute_error(test, forecast_values)
rmse = np.sqrt(mean_squared_error(test, forecast_values))
mape = np.mean(np.abs((test - forecast_values) / test)) * 100

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

# Plot residuals
residuals = test - forecast_values
plt.figure(figsize=(12, 6))
plt.plot(residuals)
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Residuals of ARIMA Forecast')
plt.xlabel('Date')
plt.ylabel('Residuals')
plt.grid(True)
plt.show()

## 7. Save Model & Forecast (Optional)

Uncomment to save the model and forecast for later use.

In [ ]:
# import joblib
# joblib.dump(model_fit, 'arima_model.pkl')
# forecast_df = pd.DataFrame({'date': forecast_dates, 'forecast': forecast_values, 'lower': forecast_ci.iloc[:, 0], 'upper': forecast_ci.iloc[:, 1]})
# forecast_df.to_csv('forecast_results.csv', index=False)

## Next Steps

To improve this pipeline:
1. Use `pmdarima.auto_arima()` for automatic parameter selection
2. Add more sophisticated models (Prophet, LSTM, etc.)
3. Implement cross-validation for time series
4. Add logging and model versioning
5. Create a proper data pipeline with feature engineering